[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Full-Text Search


## What you will be able to do

Search text for words with SQLite's FTS5 extension. Create a full-text index, find rows with
`MATCH` by words, phrases and prefixes combined with `AND`, `OR`, `NOT` and `NEAR`, rank the results
by relevance with `bm25`, and show where the words matched with `highlight` and `snippet`. Keep an
index in step with the table whose text it indexes, and turn whatever a user types into a search box
into a query that cannot raise a syntax error.


## The idea

### The problem

The stations keep a logbook: a technician's note for every station and every day, 1,460 notes for
the year, saying what the weather did and what was repaired. Which notes mention ice on the
anemometer? Which days did somebody replace a battery? A question like that is about words, and
SQL's `LIKE` does not know what a word is. `LIKE '%ice%'` finds every note containing those three
letters, so it also finds `service` and `twice`. It cannot find `heater` and `checked` close
together, cannot put the note that matches best first, and reads every note to answer.

What a search box needs is different. The words of the query, in any case, wherever they appear in
the text, with the notes that match best at the top and the matching words marked, fast enough to run
on every keystroke. And the box has to survive what people type into it, since a search language has
operators, and a user who types `AND` at the end of a query, a hyphen or an unbalanced quote should
get results, not an error.

### What full-text search is

> A search of **full text** finds rows by the words in their text. **FTS5** is SQLite's full-text
> search extension: a **virtual table**, created with `CREATE VIRTUAL TABLE ... USING fts5(...)`,
> that looks like a table of text columns and keeps an **inverted index** listing, for every word,
> the rows it appears in. A **tokenizer** decides what a word is: `unicode61`, the default, splits
> text at anything that is not a letter or a digit and ignores case. **`MATCH`** asks the index for
> the rows that contain a query's words, phrases or prefixes, combined with `AND`, `OR`, `NOT` and
> `NEAR`. **`bm25`**, the score the table's hidden `rank` column holds by default, measures how well
> a row matches, and **`highlight`** and **`snippet`** mark the words that matched.

### Why it works that way

- **An inverted index is sorted by word.** Finding the notes that contain `battery` is a search of
  the index for one word, however many notes there are, where `LIKE` reads every note's text.
- **Words, not letters.** `ice` matches the word ice, never `service` or `twice`. It also never
  matches `ices`: a plural is a different word unless the table uses the `porter` tokenizer, which
  reduces English words to a shared stem.
- **`MATCH` belongs to the virtual table.** An ordinary table has no index of words for `MATCH` to
  consult, and SQLite raises an error once it has a row to test.
- **A query is a small language.** Quotes make a phrase, `*` a prefix, a word followed by a colon
  names a column, and `AND`, `OR`, `NOT` and `NEAR` are operators, so text from a search box can be
  a syntax error. Quoting every word the user typed makes each one a plain word.
- **Lower `bm25` is better.** `bm25` gives more weight to rare words and to short texts, and FTS5
  returns it negated, so that the best match has the lowest score and `ORDER BY rank` puts it first.
- **An index can borrow its text.** With `content=` set, an FTS5 table stores only its index and
  reads the text from an ordinary table. Nothing then changes the index when that table changes,
  unless triggers do.

### Where this shows up

PostgreSQL, in the **asyncpg and psycopg3, Deep Dive** guide, has full-text search of its own, with
`tsvector`, `tsquery` and `ts_rank`. The **Peewee, Deep Dive** guide's `FTS5Model` wraps this same
virtual table in a model. The **LlamaIndex, Deep Dive** and **Haystack, Deep Dive** guides rank
documents with BM25 beside vector search, since a rare word in a query is a strong clue that meaning
alone can miss, and search servers such as Elasticsearch rank with BM25 by default. In this guide,
the **A Searchable Archive** notebook builds a searchable file from a folder of documents with FTS5.

### What this notebook covers

- `LIKE` for words, and what it finds that it should not
- `CREATE VIRTUAL TABLE ... USING fts5`, the tables behind it, and `MATCH`
- Words, case and plurals, and the `porter` tokenizer
- Phrases, prefixes, `AND`, `OR`, `NOT`, `NEAR` and column filters
- Ranking with `bm25` and `rank`, and weights for columns
- `highlight` and `snippet`
- An external content table, kept in step with its table by triggers
- Text from a search box, quoted so that it cannot be a syntax error
- When to use `LIKE`, FTS5, or FTS5 with `porter`
- A search function for the logbook
- Seven errors: `MATCH` on an ordinary table, `bm25` of the wrong table, a query ending in `AND`, a
  hyphen in a search box, `AND NOT`, the worst match first, and an index out of step with its table

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE VIRTUAL TABLE notes USING fts5(note)")
conn.executemany("INSERT INTO notes (note) VALUES (?)", [
    ("Ice on the anemometer, cleared by noon.",),
    ("Annual service of the station completed.",),
    ("Visited twice to clear ice from the mast, the gauge and the fence.",),
])
print("LIKE: ", [row[0] for row in conn.execute("SELECT rowid FROM notes WHERE note LIKE '%ice%'")])
matches = conn.execute("SELECT rowid FROM notes WHERE notes MATCH 'ice' ORDER BY rank")
print("MATCH:", [row[0] for row in matches])
conn.close()
```

```
LIKE:  [1, 2, 3]
MATCH: [1, 3]
```

`LIKE` found three notes, because `service` and `twice` contain the letters `ice`. `MATCH` found the
two notes with the word ice in them, and `ORDER BY rank` put the shorter one first, since the word is
a larger part of it.


## Setup

Six imports, and the logbook: a note for every station and day of 2025, written from that day's
readings, in an ordinary table in `logbook.db`.

- `sqlite3` holds the logbook and its full-text indexes
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook
  made, which the notes describe
- `re` finds whole words in Python, to check what `LIKE` found, in the worked examples
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end

`logbook` writes every note from the day's lowest and highest reading, and adds one of `EVENTS` on
some days, chosen by a formula, so that every note is the same everywhere this notebook runs. The
last lines check that this SQLite has FTS5, which nearly every build does.


In [1]:
import math
import re
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "logbook.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
EVENTS = [
    "Heater on the sensor mast checked and working.",
    "Battery replaced after a low voltage warning.",
    "Snow cleared from the rain gauge.",
    "Sensor recalibrated against the reference thermometer.",
    "Ice on the anemometer, so the wind readings for the morning are unreliable.",
    "Annual service of the station completed.",
    "Fence repaired after a storm.",
    "Data logger restarted after a power cut, and no readings were lost.",
    "Heaters on the mast replaced.",
    "Visited twice to check the heater.",
]
ON_WARM_DAYS = {EVENTS[2]: "Grass cut around the rain gauge.", EVENTS[4]: "Anemometer bearings greased."}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def logbook():
    """A technician's note for every station and day of 2025, written from that day's readings."""
    days = {}
    for station, hour, celsius in year_of_readings():
        days.setdefault((hour[:10], station), []).append(celsius)
    for (day, station), temperatures in sorted(days.items()):
        known = [celsius for celsius in temperatures if celsius is not None]
        if not known:
            yield station, day, "Data logger failed overnight, and there are no readings for the whole day."
            continue
        low, high = min(known), max(known)
        if high < 0:
            weather = f"Frost all day, between {low} and {high} degrees."
        elif low < 0:
            weather = f"Night frost down to {low} degrees, and a thaw to {high} by the afternoon."
        else:
            weather = f"Above freezing all day, between {low} and {high} degrees."
        day_of_year = datetime.strptime(day, "%Y-%m-%d").timetuple().tm_yday
        number = (day_of_year * 37 + list(STATIONS).index(station) * 101) % 23
        event = EVENTS[number] if number < len(EVENTS) else ""
        if low >= 0:
            event = ON_WARM_DAYS.get(event, event)
        yield station, day, f"{weather} {event}".strip()


build = sqlite3.connect(DATABASE)
build.execute("CREATE TABLE logbook (id INTEGER PRIMARY KEY, station TEXT NOT NULL, day TEXT NOT NULL, note TEXT NOT NULL)")
build.executemany("INSERT INTO logbook (station, day, note) VALUES (?, ?, ?)", logbook())
build.commit()
build.close()

check = sqlite3.connect(":memory:")
try:
    check.execute("CREATE VIRTUAL TABLE check_fts5 USING fts5(text)")
except sqlite3.OperationalError as error:
    raise RuntimeError(f"this SQLite was built without FTS5: {error}") from None
finally:
    check.close()

conn = sqlite3.connect(DATABASE)
print("notes in the logbook:", conn.execute("SELECT COUNT(*) FROM logbook").fetchone()[0])
print(conn.execute("SELECT station, day, note FROM logbook WHERE id = 6").fetchone())


notes in the logbook: 1460
('Oslo', '2025-01-02', 'Night frost down to -5.5 degrees, and a thaw to 1.4 by the afternoon.')


## Worked examples

### LIKE, and what it finds

`LIKE` compares letters. Here it looks for `ice`, and Python's `re` then checks which of the notes it
found really contain the word, `\b` marking the edge of a word:


In [2]:
found = [note for (note,) in conn.execute("SELECT note FROM logbook WHERE note LIKE '%ice%'")]
without_the_word = [note for note in found if not re.search(r"\bice\b", note, re.IGNORECASE)]

print("notes LIKE '%ice%':", len(found))
print("of them without the word ice:", len(without_the_word))
print(without_the_word[0])
print(without_the_word[1])


notes LIKE '%ice%': 156
of them without the word ice: 126
Frost all day, between -16.4 and -9.9 degrees. Visited twice to check the heater.
Night frost down to -3.9 degrees, and a thaw to 3.0 by the afternoon. Annual service of the station completed.


156 notes, and 126 of them say nothing about ice: `LIKE` found `service` and `twice`. A pattern with
spaces around the word would miss the word at the start of a note or before a comma, and no pattern
can put the best note first. `LIKE` with a leading `%` also reads every row, as the **Indexes and
Query Plans** notebook showed.

### A full-text index, and MATCH

`CREATE VIRTUAL TABLE ... USING fts5` makes a table whose columns are indexed word by word.
`UNINDEXED` keeps a column's value without indexing its words, which suits the day. The notes are
copied in with the logbook's ids as rowids, so that a match leads back to its row:


In [3]:
conn.execute("CREATE VIRTUAL TABLE search USING fts5(station, day UNINDEXED, note)")
with conn:
    conn.execute("INSERT INTO search (rowid, station, day, note) SELECT id, station, day, note FROM logbook")

print("notes MATCH 'ice':", conn.execute("SELECT COUNT(*) FROM search WHERE search MATCH 'ice'").fetchone()[0])
print(conn.execute("SELECT day, station, note FROM search WHERE search MATCH 'ice' ORDER BY rowid").fetchone())
tables = conn.execute("SELECT name FROM sqlite_schema WHERE type = 'table' ORDER BY name").fetchall()
print("tables in the file:", [name for (name,) in tables])


notes MATCH 'ice': 30
('2025-01-20', 'Bergen', 'Night frost down to -4.8 degrees, and a thaw to 2.7 by the afternoon. Ice on the anemometer, so the wind readings for the morning are unreliable.')
tables in the file: ['logbook', 'search', 'search_config', 'search_content', 'search_data', 'search_docsize', 'search_idx']


30 notes contain the word ice, and no others. `WHERE search MATCH 'ice'` names the table on the left
of `MATCH`, which searches every indexed column. The virtual table keeps its index in ordinary
tables of its own, `search_data`, `search_idx`, `search_content`, `search_docsize` and
`search_config`, which FTS5 manages and nothing else should write to.

### Words, case and plurals

The `unicode61` tokenizer ignores case and punctuation, and treats a plural as a different word. The
`porter` tokenizer, written in front of `unicode61`, reduces English words to their stems first:


In [4]:
def count(table, query):
    """How many rows of a full-text table match a query."""
    return conn.execute(f"SELECT COUNT(*) FROM {table} WHERE {table} MATCH ?", (query,)).fetchone()[0]


for query in ["heater", "HEATER", "heaters", "check"]:
    print(f"search MATCH {query!r}:", count("search", query))

conn.execute("CREATE VIRTUAL TABLE stemmed USING fts5(note, tokenize = 'porter unicode61')")
with conn:
    conn.execute("INSERT INTO stemmed (rowid, note) SELECT id, note FROM logbook")
for query in ["heater", "check"]:
    print(f"stemmed MATCH {query!r}:", count("stemmed", query))


search MATCH 'heater': 125
search MATCH 'HEATER': 125
search MATCH 'heaters': 64
search MATCH 'check': 62
stemmed MATCH 'heater': 189
stemmed MATCH 'check': 125


`heater` and `HEATER` are the same word to the index, and `heaters` is another: 125 notes against
64. `check` matched notes that say `check`, and no note that only says `checked`. With `porter`,
`heater` and `heaters` share a stem, as do `check` and `checked`, so the searches found 189 and 125
notes. A
stemmer works on English and makes mistakes, so it suits free text more than codes and names. The
table names in `count` come from this notebook's code, never from input, and the query goes in
through a placeholder.

### Phrases, prefixes and combinations

The query in `MATCH` is written in FTS5's own small language:


In [5]:
for query in [
    '"data logger"',
    "recalib*",
    "fence OR storm",
    "frost NOT heater",
    "NEAR(snow gauge, 5)",
    "station: svalbard AND logger",
]:
    print(f"{count('search', query):>5}  {query}")

print(conn.execute("SELECT COUNT(*) FROM search WHERE note MATCH 'tromso'").fetchone()[0], " note MATCH 'tromso'")


   65  "data logger"
   64  recalib*
   64  fence OR storm
  688  frost NOT heater
   33  NEAR(snow gauge, 5)
   17  station: svalbard AND logger
0  note MATCH 'tromso'


Double quotes make a phrase, so `"data logger"` needs the two words together and in order.
`recalib*` matches every word that starts with `recalib`. Words written next to each other must all
appear, as if joined by `AND`, `OR` accepts either, and `NOT` removes the rows that match what
follows it. `NEAR(snow gauge, 5)` wants both words with at most five words between them. `station:`
limits a word to one column, and so does naming the column on the left of `MATCH`: no note's text
mentions Tromso, so `note MATCH 'tromso'` finds nothing.

### Ranking with bm25

`bm25` scores every match, and `rank` is that score by default. A search for `heater OR mast`, best
first:


In [6]:
for day, station, score, note in conn.execute("""
    SELECT day, station, round(rank, 3), note FROM search
    WHERE search MATCH 'heater OR mast'
    ORDER BY rank, rowid
    LIMIT 4
"""):
    print(day, station, score, note)


2025-01-02 Svalbard -4.43 Frost all day, between -16.6 and -9.5 degrees. Heater on the sensor mast checked and working.
2025-01-03 Tromso -4.43 Frost all day, between -8.9 and -1.4 degrees. Heater on the sensor mast checked and working.
2025-01-25 Svalbard -4.43 Frost all day, between -17.1 and -10.2 degrees. Heater on the sensor mast checked and working.
2025-01-26 Tromso -4.43 Frost all day, between -9.0 and -2.4 degrees. Heater on the sensor mast checked and working.


The notes that contain both `heater` and `mast` come first, with the lowest scores. A word that
matches twice counts for more than one that matches once, a rare word for more than a common one,
and a short note for more than a long one. Many notes here are the same sentence, so `rowid` settles
the order among equal scores. The numbers themselves only rank the notes of one query: they mean
nothing compared across queries.

`bm25` also takes a weight for every column, in the order the table declares them:
`bm25(search, 10.0, 1.0, 1.0)` would count a match in `station` ten times as much as one in `note`,
and the weight for `day` would do nothing, since that column is not indexed.

### highlight and snippet

`highlight` returns a column's text with every matched word wrapped in the markers given, and
`snippet` returns only the part of the text around the matches, cut to a number of words:


In [7]:
for highlighted, snipped in conn.execute("""
    SELECT highlight(search, 2, '[', ']'), snippet(search, 2, '[', ']', '...', 6)
    FROM search WHERE search MATCH 'reference OR thermometer'
    ORDER BY rank, rowid LIMIT 2
"""):
    print(highlighted)
    print(snipped)


Frost all day, between -17.2 and -10.1 degrees. Sensor recalibrated against the [reference] [thermometer].
...Sensor recalibrated against the [reference] [thermometer].
Frost all day, between -8.9 and -2.3 degrees. Sensor recalibrated against the [reference] [thermometer].
...Sensor recalibrated against the [reference] [thermometer].


The `2` is the column's position, counting from 0, so it names `note`. `highlight` suits short text,
and `snippet` suits long documents, where a few words around the match are all a result list can
show. In a web page the markers would be HTML tags, around text that has itself been escaped.

### An external content table, kept in step

`search` holds a second copy of every note. With `content='logbook'`, an FTS5 table stores only its
index and reads the text from the logbook itself, and `content_rowid` names the column its rowids
come from. FTS5's documentation gives three triggers that keep such an index in step with its table:


In [8]:
conn.executescript("""
    CREATE VIRTUAL TABLE logbook_index USING fts5(note, content = 'logbook', content_rowid = 'id');
    INSERT INTO logbook_index (logbook_index) VALUES ('rebuild');

    CREATE TRIGGER logbook_after_insert AFTER INSERT ON logbook BEGIN
        INSERT INTO logbook_index (rowid, note) VALUES (new.id, new.note);
    END;
    CREATE TRIGGER logbook_after_delete AFTER DELETE ON logbook BEGIN
        INSERT INTO logbook_index (logbook_index, rowid, note) VALUES ('delete', old.id, old.note);
    END;
    CREATE TRIGGER logbook_after_update AFTER UPDATE ON logbook BEGIN
        INSERT INTO logbook_index (logbook_index, rowid, note) VALUES ('delete', old.id, old.note);
        INSERT INTO logbook_index (rowid, note) VALUES (new.id, new.note);
    END;
""")

with conn:
    conn.execute("UPDATE logbook SET note = note || ' Lightning damage to the fence.' WHERE id = ?", (4 * 225 + 4,))
    conn.execute("INSERT INTO logbook (station, day, note) VALUES (?, ?, ?)",
                 ("Kirkenes", "2025-12-31", "Site surveyed for a new station. Lightning mast planned."))
print(conn.execute("""
    SELECT logbook.station, logbook.day FROM logbook_index JOIN logbook ON logbook.id = logbook_index.rowid
    WHERE logbook_index MATCH 'lightning' ORDER BY logbook.id
""").fetchall())


[('Tromso', '2025-08-14'), ('Kirkenes', '2025-12-31')]


`'rebuild'`, inserted into the table's own name, is a command: it read every note from the logbook
and built the index. After that, the triggers kept it in step: the updated note and the new one were
both found. A `'delete'` row gives FTS5 the old text, which it needs in order to take the old words
out of the index, since it no longer stores the text itself. The join brings back the station and
day from the logbook through the rowid.

### Text from a search box

What a user types is a query in FTS5's language whether they meant it or not. `as_words` turns any
text into a query of plain words, each quoted as a phrase of one word, with any quote inside a word
doubled:


In [9]:
def as_words(text):
    """Text typed into a search box, as an FTS5 query that needs every word and treats none as an operator."""
    return " ".join('"' + word.replace('"', '""') + '"' for word in text.split())


for typed in ["heater AND", '"heater', "data-logger", "place: ice", "power cut"]:
    query = as_words(typed)
    print(f"{typed!r:<16} -> {query!r:<28} {count('logbook_index', query)} notes")


'heater AND'     -> '"heater" "AND"'             125 notes
'"heater'        -> '"""heater"'                 125 notes
'data-logger'    -> '"data-logger"'              65 notes
'place: ice'     -> '"place:" "ice"'             0 notes
'power cut'      -> '"power" "cut"'              64 notes


Every input became a query that runs. `AND` became the word and, which every note's weather
sentence contains, so the first search found the 125 notes about a heater. The lone quote was doubled
inside a phrase, and the tokenizer drops quotes as it drops all punctuation, so the second search was
for heater too. `data-logger` became a phrase, in which the tokenizer splits the hyphen as it did in
the notes, so it found the 65 notes about the data logger. `place:` no longer names a column, and no
note contains the word place. The cost is that a user can no longer use the operators on purpose,
which most search boxes can do without.

### LIKE, FTS5, or FTS5 with porter

| Write | When | Why |
|---|---|---|
| `LIKE '%text%'` | a few rows, or letters inside codes and identifiers, such as part of a serial number | it compares characters, needs no index, and finds text inside words |
| an FTS5 table with the default `unicode61` tokenizer | searching names, codes or text in any language by whole words | it finds exact words fast, in any case, and ranks them with `bm25` |
| an FTS5 table with `tokenize = 'porter unicode61'` | searching English prose, where `repair`, `repairs` and `repaired` mean the same | it matches every word by its stem, at the cost of some wrong matches |
| `content = 'table'` with triggers | text that already lives in an ordinary table | the text is stored once, and the triggers keep the index in step |

The default for a search box over text is an FTS5 table with its own content, and `as_words` around
whatever the user typed.

### A search function for the logbook

The pieces of this notebook in one function: plain words from a search box, an optional station, the
best matches first, with a snippet of every note. It searches the external content index and reads
the station and day from the logbook:


In [10]:
def search_logbook(text, station=None, limit=3):
    """The notes that best match the words in text, optionally at one station, with a snippet of each."""
    sql = """
        SELECT logbook.day, logbook.station, snippet(logbook_index, 0, '[', ']', '...', 8)
        FROM logbook_index JOIN logbook ON logbook.id = logbook_index.rowid
        WHERE logbook_index MATCH ? AND (? IS NULL OR logbook.station = ?)
        ORDER BY logbook_index.rank, logbook.id
        LIMIT ?
    """
    return conn.execute(sql, (as_words(text), station, station, limit)).fetchall()


for day, station, snipped in search_logbook("heater mast"):
    print(day, station, snipped)
print()
for day, station, snipped in search_logbook("data-logger", station="Svalbard"):
    print(day, station, snipped)


2025-01-02 Svalbard ...[Heater] on the sensor [mast] checked and working.
2025-01-03 Tromso ...[Heater] on the sensor [mast] checked and working.
2025-01-25 Svalbard ...[Heater] on the sensor [mast] checked and working.

2025-03-02 Svalbard [Data logger] failed overnight, and there are no...
2025-01-14 Svalbard ...10.1 degrees. [Data logger] restarted after a...
2025-02-06 Svalbard ...9.5 degrees. [Data logger] restarted after a...


The first search needed both words, found the notes about the heater on the mast, and showed each
one cut around its matches. The second came from a hyphenated search box entry, limited to one
station, and found Svalbard's notes about the data logger, the failure on 2 March among them. The
condition on the station, true whenever the station passed is `NULL`, lets one statement serve both
calls, so the SQL never changes with the input.

### Where each part came from

| In the search function | What it relies on | The section that showed it |
|---|---|---|
| `logbook_index`, with `content = 'logbook'` | an index that reads its text from the logbook | An external content table, kept in step |
| `MATCH ?` with `as_words(text)` | a search box's text as plain words | Text from a search box |
| `ORDER BY logbook_index.rank, logbook.id` | the best match first, and a fixed order among equals | Ranking with bm25 |
| `snippet(logbook_index, 0, ...)` | the text around the matches | highlight and snippet |
| `JOIN logbook ON logbook.id = logbook_index.rowid` | a match leading back to its row | A full-text index, and MATCH |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/16-full-text-search-solutions.ipynb).

**1.** Find the three notes that best match `battery OR power`, and print each one's day, station and
score, rounded to three places.


In [11]:
# your code here


**2.** Count the notes in which `snow` comes within two words of `gauge`, and within five.


In [12]:
# your code here


**3.** Using a column filter, count the notes about frost at each station, highest count first.


In [13]:
# your code here


**4.** Count the notes that match `repair` in `search` and in `stemmed`, and explain the difference.


In [14]:
# your code here


**5.** Insert a note into the logbook, find it through `logbook_index`, delete it, and show the
search no longer finds it.


In [15]:
# your code here


**6.** Create an `fts5vocab` table over `search` and print the five words that appear in the most
notes, with how many notes contain each.


In [16]:
# your code here


## Common errors

### sqlite3.OperationalError: unable to use function MATCH in the requested context


In [17]:
conn.execute("SELECT day, station FROM logbook WHERE note MATCH 'ice'").fetchall()


OperationalError: unable to use function MATCH in the requested context

`logbook` is an ordinary table, and `MATCH` works only against a full-text table, which has the
index of words to answer it. SQLite raises the error when it has a row to test, so the same query
against an empty ordinary table returns an empty list without a word, and the mistake can hide until
data arrives. Ask the full-text table:


In [18]:
print(conn.execute("SELECT day, station FROM search WHERE search MATCH 'ice' ORDER BY rank, rowid LIMIT 2").fetchall())


[('2025-01-22', 'Svalbard'), ('2025-01-23', 'Tromso')]


### sqlite3.OperationalError: no such column: logbook


In [19]:
conn.execute("SELECT day, bm25(logbook) FROM logbook WHERE id = 1").fetchall()


OperationalError: no such column: logbook

`bm25` takes the full-text table itself as its first argument, and in a query on an ordinary table
that name means nothing, so SQLite read `logbook` as a column. `bm25`, `highlight` and `snippet` work
only in a query whose `MATCH` is on the table they name:


In [20]:
print(conn.execute("""
    SELECT day, round(bm25(search), 3) FROM search WHERE search MATCH 'ice' ORDER BY rank, rowid LIMIT 1
""").fetchall())


[('2025-01-22', -3.239)]


### sqlite3.OperationalError: fts5: syntax error near ""


In [21]:
typed = "heater AND"
conn.execute("SELECT rowid FROM search WHERE search MATCH ?", (typed,)).fetchall()


OperationalError: fts5: syntax error near ""

The search box's text went to `MATCH` as a query, and `AND` at the end is an operator with nothing
after it, so FTS5 reached the end of the text, the empty `""` in the message, expecting a word. An
unbalanced double quote fails the same way, with `unterminated string`. Pass the text through
`as_words`, which quotes every word:


In [22]:
print(conn.execute("SELECT COUNT(*) FROM search WHERE search MATCH ?", (as_words(typed),)).fetchone()[0], "notes")


125 notes


### sqlite3.OperationalError: no such column: logger


In [23]:
typed = "data-logger"
conn.execute("SELECT rowid FROM search WHERE search MATCH ?", (typed,)).fetchall()


OperationalError: no such column: logger

In FTS5's language, a hyphen in front of a word is part of the syntax for columns, so `-logger` was
read as a column called `logger`, which the table does not have. Quoted, the same text is a phrase of
two words, which is what the user meant:


In [24]:
print(conn.execute("SELECT COUNT(*) FROM search WHERE search MATCH ?", (as_words(typed),)).fetchone()[0], "notes")


65 notes


### sqlite3.OperationalError: fts5: syntax error near "NOT"


In [25]:
conn.execute("SELECT COUNT(*) FROM search WHERE search MATCH 'frost AND NOT heater'").fetchone()


OperationalError: fts5: syntax error near "NOT"

In SQL, `AND NOT` is how a condition excludes something. In FTS5, `NOT` joins two queries by itself,
the rows that match the first but not the second, so `AND NOT` puts two operators in a row. Write
`NOT` alone:


In [26]:
print(conn.execute("SELECT COUNT(*) FROM search WHERE search MATCH 'frost NOT heater'").fetchone()[0], "notes")


688 notes


### No error, and the weakest match first: ORDER BY bm25 DESC


In [27]:
print(conn.execute("""
    SELECT day, station, note FROM search WHERE search MATCH 'heater OR mast'
    ORDER BY bm25(search) DESC, rowid LIMIT 1
""").fetchone())


('2025-01-22', 'Bergen', 'Night frost down to -4.2 degrees, and a thaw to 2.4 by the afternoon. Visited twice to check the heater.')


The first result mentions the heater once and has no mast in it. A higher score usually means a
better match, and `DESC` puts the highest first, but FTS5 negates `bm25`, so the best match has the
lowest score, and `DESC` put the weakest match at the top. Sort ascending, or by `rank`, which is the
same number:


In [28]:
print(conn.execute("""
    SELECT day, station, note FROM search WHERE search MATCH 'heater OR mast'
    ORDER BY rank, rowid LIMIT 1
""").fetchone())


('2025-01-02', 'Svalbard', 'Frost all day, between -16.6 and -9.5 degrees. Heater on the sensor mast checked and working.')


### No error, and a changed note that search cannot find: an external content table with no triggers


In [29]:
conn.execute("CREATE VIRTUAL TABLE untriggered USING fts5(note, content = 'logbook', content_rowid = 'id')")
with conn:
    conn.execute("INSERT INTO untriggered (untriggered) VALUES ('rebuild')")
    conn.execute("UPDATE logbook SET note = note || ' Hail damage to the rain gauge.' WHERE id = ?", (4 * 181 + 2,))

print("untriggered MATCH 'hail':", count("untriggered", "hail"))
print("logbook_index MATCH 'hail':", count("logbook_index", "hail"))


untriggered MATCH 'hail': 0
logbook_index MATCH 'hail': 1


The note was changed, and the index built before the change knows nothing of it, so the search found
no hail. `logbook_index` found it, because its triggers ran on the update. An external content index
stores no text of its own, so it cannot notice a change, and without triggers its words drift away
from the notes they point at: new words are missing, and old words keep finding notes that no longer
say them. Rebuild the index to catch up, and add the triggers to stay caught up:


In [30]:
with conn:
    conn.execute("INSERT INTO untriggered (untriggered) VALUES ('rebuild')")
print("untriggered MATCH 'hail', after rebuild:", count("untriggered", "hail"))
conn.close()


untriggered MATCH 'hail', after rebuild: 1


Last, the connection is closed, so this cell removes the scratch folder, with the database in it:


In [31]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `LIKE` compares letters, so it finds words inside other words, cannot rank, and reads every row.
- `CREATE VIRTUAL TABLE ... USING fts5` builds an index of words, and `MATCH` searches it, in any
  case, for whole words.
- A query can use phrases in double quotes, prefixes with `*`, `AND`, `OR`, `NOT`, `NEAR` and column
  filters, so text from a search box has to be quoted word by word before it reaches `MATCH`.
- `bm25` is negated, so the lowest score is the best match, and `ORDER BY rank` sorts best first.
- `highlight` and `snippet` mark where the words matched, by the column's position.
- The `porter` tokenizer matches English words by their stems, and `unicode61` matches them exactly.
- An external content table stores its text once, in an ordinary table, and needs triggers, or a
  `'rebuild'`, to stay in step with it.


## What is next

The **Concurrency and WAL** notebook lets more than one connection use a database at once: what
`database is locked` means, how long a connection waits with `busy_timeout`, and the write-ahead log,
which lets readers carry on while a writer writes, and still allows only one writer.


---

&#8592; **Previous:** [Indexes and Query Plans](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/15-indexes-and-query-plans.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
